### <u>Notes:</u>

<p>This notebook script contains some error messages at the bottom. It is due to the conflicts between Jupyter Notebook and my python interpreter, which is yet to be solved. Having said that, all of the cells should be runnable in every OS.</p>

In [3]:
import os 
import glob

import numpy as np 
from shutil import rmtree 

In [4]:
def clean():
    for name in ("my-experiment", "ioh_data"):
        for path in glob.glob(f"{name}*"):
            if os.path.isfile(path):
                os.remove(path)
            if os.path.isdir(path):
                rmtree(path, ignore_errors=True)


def ls(p="./"):
    for obj in os.listdir(os.path.normpath(p)):
        print(obj)


def cat(f):
    with open(os.path.normpath(f)) as h:
        print(h.read())


clean()

In [5]:
import ioh 
from ioh import iohcpp

In [6]:
# a list of problems can be accessed via the base classes 
real_problems: dict[int, str] = ioh.problem.RealSingleObjective.problems
print(real_problems)

{1: 'Sphere', 2: 'Ellipsoid', 3: 'Rastrigin', 4: 'BuecheRastrigin', 5: 'LinearSlope', 6: 'AttractiveSector', 7: 'StepEllipsoid', 8: 'Rosenbrock', 9: 'RosenbrockRotated', 10: 'EllipsoidRotated', 11: 'Discus', 12: 'BentCigar', 13: 'SharpRidge', 14: 'DifferentPowers', 15: 'RastriginRotated', 16: 'Weierstrass', 17: 'Schaffers10', 18: 'Schaffers1000', 19: 'GriewankRosenbrock', 20: 'Schwefel', 21: 'Gallagher101', 22: 'Gallagher21', 23: 'Katsuura', 24: 'LunacekBiRastrigin', 30: 'UniformStarDiscrepancy10', 31: 'UniformStarDiscrepancy25', 32: 'UniformStarDiscrepancy50', 33: 'UniformStarDiscrepancy100', 34: 'UniformStarDiscrepancy150', 35: 'UniformStarDiscrepancy200', 36: 'UniformStarDiscrepancy250', 37: 'UniformStarDiscrepancy500', 38: 'UniformStarDiscrepancy750', 39: 'UniformStarDiscrepancy1000', 40: 'SobolStarDiscrepancy10', 41: 'SobolStarDiscrepancy25', 42: 'SobolStarDiscrepancy50', 43: 'SobolStarDiscrepancy100', 44: 'SobolStarDiscrepancy150', 45: 'SobolStarDiscrepancy200', 46: 'SobolStarDis

In [8]:
# in order to instantiate a problem instance, we can do the following: 

function_id = real_problems[1]


problem = ioh.get_problem(
    fid = function_id, 
    instance=1,
    dimension = 10, 
    problem_class = ioh.ProblemClass.REAL  # pyright: ignore[reportCallIssue]
)

problem 

<RealSingleObjectiveProblem 1. Sphere (iid=1 dim=10)>

In [9]:
# the problem class includes information about the problem, which can be retrieved via the meta data accessor 
problem.meta_data.name 

'Sphere'

In [10]:
# the current state of the problem, e.g., the number of evaluations, best seen points, etc. are stored in the problem state 
problem.state 

<State evaluations: 0 final_target_found: false current_best: <Solution x: [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan] y: inf>>

In [11]:
# every problem has a simple box-bound assocaited 
problem.bounds 

<BoxConstraint lb: [[-5, -5, -5, -5, -5, -5, -5, -5, -5, -5]] ub: [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]>

In [12]:
# we can access the constraint information of the problem 

x0 = np.random.uniform(problem.bounds.lb, problem.bounds.ub)


# evaluation happens like a 'normal' objective function would 
problem(x0)

print("-----------------------")

# whenever the problem is evaluated, the state changes 
problem.state 

-----------------------


<State evaluations: 1 final_target_found: false current_best: <Solution x: [-1.8980657759420594, -1.2669802913587134, -4.74591031436586, -4.266511513513032, 3.6082409636079795, -3.407904561557914, -0.04815905901603301, 3.4964171447758474, -4.5574418197525794, 0.03348247685286587] y: 222.444175172921>>

In [13]:
# additionally, it is possible to evaluate a list of points 

n_points: int = 5 
X = np.random.uniform(problem.bounds.lb, problem.bounds.ub, size=(n_points, problem.meta_data.n_variables))

In [14]:
# if we want to perform multiple runs with the same objective function, after every run, the problem has to be reset 

def run_experiment(problem, algorithm, n_runs=5):
    for run in range(n_runs):


        # run the algorithm on the problem 
        algorithm(problem)

        # print the best found for this run 
        print(f"run: {run + 1} - best found: {problem.state.current_best.y: .3f}")


        # reset the problem 
        problem.reset()

In [15]:
class RandomSearch:
    def __init__(self, n: int, length: float = 0.0):
        self.n: int = n 
        self.length: float = length 
    


    def __call__(self, problem: ioh.problem.RealSingleObjective) -> None:
        # evaluate the problem n times with a randomly generated solution


        for _ in range(self.n):
            # we can use the problems bound accessor to get information about the problem bounds 
            x = np.random.uniform(problem.bounds.lb, problem.bounds.ub)
            self.length = float(np.linalg.norm(x))


            problem(x)
            

In [16]:
# using the random search algorithm, we can then run a simple experiment 
run_experiment(problem, RandomSearch(10))

run: 1 - best found:  159.223
run: 2 - best found:  105.869
run: 3 - best found:  154.549
run: 4 - best found:  117.492
run: 5 - best found:  119.831


In [20]:
def styblinsky_tang(x: np.ndarray) -> float:
    return float(np.sum(np.power(x, 4) - (16 * np.power(x, 2)) + (5 * x)) / 2)


X = np.array([-2.903534]*10)

styblinsky_tang(X) # global minima

-391.661657037714

In [21]:
# we can wrap this function in ioh, as this is a continuous function, we use wrap_real_problem()

ioh.problem.wrap_real_problem(
    f=styblinsky_tang, # handle to the function 
    name="StyblinskiTang", # name to be used when instantiating 
    optimization_type=ioh.OptimizationType.MIN, # specify that we want to minimize  # pyright: ignore[reportAttributeAccessIssue]
    lb=-5,                  # lower bound 
    ub=5,                   # upper bound 
)


In [19]:
# we can create an instance of this problem., wrapped in ioh 
problem = ioh.get_problem("StyblinskiTang", dimension=10)
problem

<RealSingleObjectiveProblem 1121. StyblinskiTang (iid=1 dim=10)>

In [23]:
problem(X.tolist())

-391.661657037714

In [24]:
# variable transformation R^d -> R^d 
def transofrm_variables(x: np.ndarray, instance_id: int) -> np.ndarray:
    c = (instance_id - 1) * 0.5 
    return x + c 



# objective transformation R -> R
def transform_objectives(y: float, instance_id: int) -> float:
    c = instance_id 
    return y * c 

# note that we can overwrite a previously defined problem by calling wrap_real_problem again 
ioh.problem.wrap_real_problem(
    f=styblinsky_tang, 
    name="StyblinskiTang",
    optimization_type=ioh.OptimizationType.MIN,  # pyright: ignore[reportAttributeAccessIssue]
    lb = -5,
    ub = 5, 

    # adding the transformation functions 
    transform_variables=transofrm_variables, 
    transform_objectives=transform_objectives
)




In [25]:
# we can not create different instances of the same problem 
fid = "StyblinskiTang"
instance1 = ioh.get_problem(fid, instance=1, dimension=10)
instance2 = ioh.get_problem(fid, instance=2, dimension=10)
instance1, instance2

(<RealSingleObjectiveProblem 1121. StyblinskiTang (iid=1 dim=10)>,
 <RealSingleObjectiveProblem 1121. StyblinskiTang (iid=2 dim=10)>)

In [26]:
# note that when evaluating the same point, each instance gives a different transformed value 
instance1(x0), instance2(x0)

(-82.31981587952981, -376.18055381694944)

### Logging data 

In [ ]:
logger = ioh.logger.Analyzer(
    root = os.getcwd(),     # store data in the current working directory  # pyright: ignore[reportArgumentType]
    folder_name = "my-experiment", # in a folder named 'my-experiment'
    algorithm_name="random-search", # meta-data for the algorithm used to generate the results 
    store_positions=True           # store x-variables in the logged files  
)


# this automatically creates a folder "my-experiment" in the current working directory 
# if the folder already exists, it will give an additional number to make the name unique 
logger 

<Analyzer /home/sething2002/2025_S2/COMP_SCI_3316_Self/ioh_experiment/my-experiment>

In [29]:
ls()

data
.venv
main.py
draft.py
README.md
Tracking_params.py
pyproject.toml
problem_example.py
.python-version
uv.lock
tutorial.ipynb
my-experiment


In [ ]:
# in order to log data for a problem, we only have to attach it to a logger 
problem = ioh.get_problem(fid, instance=1, dimension=2)
problem.attach_logger(logger)


# we can then run the random search as before, only now all data will be logged to a file 
run_experiment(problem, RandomSearch(10), n_runs=1)


# The close() method forces our logger to flush and write files (> v0.3.3)
# This is only required when using the logger in an interactive context, such as this jupyter notebook 
# otherwise, the destructor, which is called on exit of the python interpreter, forces such a write automatically 
logger.close() 

run: 1 - best found: -64.021


In [31]:
StyblinskiTang_function_identifier = max(ioh.problem.RealSingleObjective.problems.keys())
StyblinskiTang_function_identifier

1121

In [32]:
cat(f"my-experiment/IOHprofiler_f{StyblinskiTang_function_identifier}_StyblinskiTang.json")

{
	"version": "0.3.18", 
	"suite": "unknown_suite", 
	"function_id": 1121, 
	"function_name": "StyblinskiTang", 
	"maximization": false, 
	"algorithm": {"name": "random-search", "info": "algorithm_info"},
	"attributes": ["evaluations", "raw_y"],
	"scenarios": [
		{"dimension": 2,
		"path": "data_f1121_StyblinskiTang/IOHprofiler_f1121_DIM2.dat",
		"runs": [
			{"instance": 1, "evals": 10, "best": {"evals": 6, "y": -64.02126519406698, "x": [-2.8173261850400944, 2.68797576619421]}}
		]}
	]
}



In [33]:
cat(f"my-experiment/data_f{StyblinskiTang_function_identifier}_StyblinskiTang/IOHprofiler_f{StyblinskiTang_function_identifier}_DIM2.dat")

evaluations raw_y x0 x1
1 -4.1449436893 0.212714 -0.600049
4 -56.3884465388 -2.682306 1.927867
6 -64.0212651941 -2.817326 2.687976
10 -34.9381242576 3.719692 -3.582610



### Trigger 


<p>The default behavior of the <i>Analyzer</i> logger is to log data only when there is an improvement of the objective value. We can change this behavior, by specifying one or more triggers, which are logical operators, which when one of them evaluates to True, will cause data to be logged.</p> <br>

<p>There is a number of trigger variants which can be used to customize the logging. In the following example, a trigger is defined which evaluates to True, every 3 function evaluations. It it combined with a trigger for improvement, so data will be logged on every 3rd function evaluation, or when there is an observed improvement of the objective value.</p>

In [37]:
triggers = [
    ioh.logger.trigger.Each(3),
    ioh.logger.trigger.OnImprovement()
]


logger = ioh.logger.Analyzer(
    root = os.getcwd(),  # pyright: ignore[reportArgumentType]
    folder_name="my-experiment",
    algorithm_name="random-search",
    store_positions=True, 


    # add the triggers to the logger 
    triggers = triggers 
)

In [ ]:
# rerun the same experiment as before 
problem = ioh.get_problem(fid, instance=1, dimension=2)
problem.attach_logger(logger)
run_experiment(problem, RandomSearch(10), n_runs=1)
logger.close()  # type: ignore

run: 1 - best found: -47.383


In [39]:
# we can see that data is logged either if there is improvement, or on every 3rd evaluation 
cat(f"my-experiment-1/data_f{StyblinskiTang_function_identifier}_StyblinskiTang/IOHprofiler_f{StyblinskiTang_function_identifier}_DIM2.dat")


evaluations raw_y x0 x1
1 -17.4137890629 -0.450162 3.485264
2 -47.3833530161 -2.107664 1.818612
3 43.1559005118 3.910080 -4.563321
6 0.1495844406 0.145795 -0.017097
9 63.6389008651 3.695173 -4.809229
10 -11.9457801997 -4.222705 -3.862597



### Standardized Experimental Setup (Python Only)

<p>In Python, we provide <code>Experiment</code> class which can be used to easily run a given algorithm over a larger number of problems. </p>

In [40]:
experiment = ioh.Experiment(
    algorithm=RandomSearch(10), # an algorithm instance 
    fids = [1, StyblinskiTang_function_identifier], # the id's of the problems we want to test 
    iids=[1, 10], # the instances 
    dims = [2, 10], # the dimensions 
    reps = 3, # the number of runs 
    zip_output= True, 
    old_logger= True  
)

In [41]:
experiment.run()

In [42]:
ls("ioh_data")

IOHprofiler_f1121_StyblinskiTang.info
data_f1_Sphere
data_f1121_StyblinskiTang
IOHprofiler_f1_Sphere.info


In [ ]:
clean() 

In [ ]:
# clean up 
rmtree("my-experiment", ignore_errors=True)
rmtree("ioh_data", ignore_errors=True)

## Working with constraints (> v0.3.3)



<p>Every problem has a set of constraints associated with it. For most of the currently implemented problems, this is an empty set by defaults. However, we can modify this set, and add arbitrary functions as constraints.</p>

In [44]:
# we can take the sphere function as example 
f_name:str = "Sphere"
p = ioh.get_problem(f_name, 1, 2)



#  the set of constrinats is empty by default 
p.constraints

<ConstraintSet: >

In [46]:
# we can turn the bounds of the problem into a constraint 
print(p.bounds)



# by default. bounds are enforced, so violating them changes nothing to the returned objective function value 
p([10, 10])

<BoxConstraint lb: [[-5, -5]] ub: [[5, 5]]>


298.96209408

In [48]:
# for bound constraints, violation is computed as the sum of swuared violation per component 
p.enforce_bounds(
    how=ioh.ConstraintEnforcement.SOFT, # the enforcement strategy: SOFT ensures penalization on violation by a penalty (y + p) # pyright: ignore[reportCallIssue] # type: ignore
    weight=1.0, # Penalty p is computed as: weight * violation ^ exponent 
    exponent= 1.0, 
)


# now the bounds are in the constraint set 
p.constraints 

<ConstraintSet: <BoxConstraint lb: [[-5, -5]] ub: [[5, 5]]>>

In [49]:
# we can now observe the added penalty term 
p([10, 10])

348.96209408

In [ ]:
# there are several strategies that modify a constraint's behavior: 
types = (
    ioh.ConstraintEnforcement.NOT, # don't calculate the constraint function  # type: ignore
    ioh.ConstraintEnforcement.HIDDEN, # calculate the constraint, but don't penalize # type: ignore
    ioh.ConstraintEnforcement.SOFT, # calculate both constraint and objective function value  # type: ignore
    ioh.ConstraintEnforcement.HARD, # calculate the constraint, if there is violation, don't calculate y, and only return p  # type: ignore
)


for strategy in types:
    p.enforce_bounds(how=strategy, weight=1.0, exponent=1.0) # type: ignore
    print(strategy, p([10, 10], p.constraints.violation())) # type: ignore

In [50]:
# note that when we use a HARD method, the objective value is actually lower (50.0) than the actual objective value (298.96)
# this is because we didn't change any other parameters, that influence the penalty. Normally, with HARD constraints, we 
# want to also make sure that we return a y-value that is infeasible. We can do this by changing the weight and/or exponent paramters. 


p.enforce_bounds(
    how=ioh.ConstraintEnforcement.HARD,  # type: ignore
    weight=float("inf"), 
    exponent=1.0 
)


# any infeasible solution gets an infinity value 
p([10, 10])

inf

In [ ]:
# note that the actual objective function is not called for infeasible points with HARD constraints: 
p.state.y_unconstrained  # type: ignore

inf

In [ ]:
# if we pass a feasible point, it is evaluated 
p([1, -1]), p.state.y_unconstrained  # type: ignore

(80.06289408, 80.06289408)

### Custom constraint functions 

In [ ]:
# we can create custom constraint functions, which can be 'just' functions that check 
# something on x.


# such functions should return a non-zero floating point value when there is violation and zero otherwise. 

# ====================== Bugs ????? =================================
# def is_x_ordered(X: np.ndarray) -> float:
#     """Checks that x_i < x_(i+1)"""

#     is_strictly_increasing = (np.diff(X) > 0).all()
#     return float(not is_strictly_increasing)

# X_1 = np.array([10, 1, 3])
# # x is not ordered, so there is a violaton (1.)
# print(is_x_ordered(X_1))

# X_2 = np.ndarray([1, 3, 10])
# # x is ordered, so there is no violation (0.)
# print(is_x_ordered(X_2))
# =======================================================

1.0
1.0


In [ ]:

# =======================================================
def is_x_ordered(x: np.ndarray) -> float:
    '''Checks that xi < xi+1'''

    return 1. - (np.diff(x) > 0).all().astype(float)

# x is not ordered, so there is a violation (1.)
print(is_x_ordered([10, 1, 3]))

# x is ordered, so there is no violation (0.)
print(is_x_ordered([1, 3, 10]))

In [78]:
# we can add such a function to an arbitrary problem as constraint like so:

constraint = ioh.RealConstraint(is_x_ordered, name="is_x_ordered", weight=100_000)


p = ioh.get_problem("Sphere", 1, 3)
p.add_constraint(constraint)



# note that it gets added to the list of constraints 
p.constraints 


<ConstraintSet: <FunctionalConstraint is_x_ordered>>

In [79]:
print(p([4,1,3])) # the penalty gets added 
print(p([1,3,4])) # the penalty doesn't get added 

112.04147008000001
100119.63347008


In [80]:
# we can remove the constraint again 
p.remove_constraint(constraint)
p.constraints 

<ConstraintSet: >

### Wrapping Constraints 

In [81]:
# we can also add constraints to the problem definition to wrap problem 
ioh.problem.wrap_real_problem(
    styblinsky_tang, 
    name="StyblinskiTang",
    optimization_type=ioh.OptimizationType.MIN, 
    lb = -5, 
    ub = 5, 

    # adding the constraints 
    constraints=[constraint]
)

In [82]:
# now if we instantiate an instance of the Styblinski-Tang function, it has the x-ordering constraint added automatically 
p = ioh.get_problem("StyblinskiTang", 1, 5)
p.constraints 

<ConstraintSet: <FunctionalConstraint is_x_ordered>>

### Logginc constraints 

In [83]:
logger = ioh.logger.Analyzer(
    root = os.getcwd(), 
    folder_name="my-experiment", 
    algorithm_name="random-search",
    store_positions=True, 

    # we add the constraint properties to the list of logged items, 
    # note that these properties are aggregated values for ALL applied constraints, 
    # so when we have more than one constraint added to a problem, these values will be
    # the sums of all the applied constraints. 

    additional_properties=[
        ioh.logger.property.CURRENTY, # the constrained y-value, by default only the untransformed and unconstrained y 
                                       # value is logged 
        ioh.logger.property.VIOLATION, # the violation value 
        ioh.logger.property.PENALTY, # the appluied penalty 
    ]
)


# we can now run the same experiment 
problem = ioh.get_problem(fid, instance=1, dimension=2)


problem.attach_logger(logger)
run_experiment(problem, RandomSearch(10), n_runs=1)
logger.close()

run: 1 - best found: -63.917


In [85]:
cat(f"my-experiment/IOHprofiler_f{StyblinskiTang_function_identifier}_StyblinskiTang.json")

{
	"version": "0.3.18", 
	"suite": "unknown_suite", 
	"function_id": 1121, 
	"function_name": "StyblinskiTang", 
	"maximization": false, 
	"algorithm": {"name": "random-search", "info": "algorithm_info"},
	"attributes": ["evaluations", "raw_y"],
	"scenarios": [
		{"dimension": 2,
		"path": "data_f1121_StyblinskiTang/IOHprofiler_f1121_DIM2.dat",
		"runs": [
			{"instance": 1, "evals": 10, "best": {"evals": 6, "y": -64.02126519406698, "x": [-2.8173261850400944, 2.68797576619421]}},
			{"instance": 1, "evals": 10, "best": {"evals": 2, "y": -32.209415728557005, "x": [2.646231946718167, 1.1878212903071939]}}
		]}
	]
}



In [87]:
cat(f"my-experiment/data_f{StyblinskiTang_function_identifier}_StyblinskiTang/IOHprofiler_f{StyblinskiTang_function_identifier}_DIM2.dat")

evaluations raw_y x0 x1
1 -4.1449436893 0.212714 -0.600049
4 -56.3884465388 -2.682306 1.927867
6 -64.0212651941 -2.817326 2.687976
10 -34.9381242576 3.719692 -3.582610
evaluations raw_y x0 x1
1 -18.6786435765 2.079950 -4.166329
2 -32.2094157286 2.646232 1.187821
10 -23.9825546993 2.265485 -0.338682

